# 🎵 Vectors2Vibes — Lyric Transcription & Vector Embedding Pipeline V3
**Pipeline:**
1. Iterates over original_dataset.csv and downloads each file from Huggingface.
2. Transcribes lyrics with OpenAI Whisper
3. Embeds transcriptions with `sentence-transformers`
4. Saves every 50 songs as a checkpoint Parquet to Google Drive

!!! **Before running:** delete any existing contents in the "vectors2vibes_output" folder to ensure the embeddings start from scratch.

**To run:**
1. Create a folder in your Drive called "vectors2vibes_output"
2. Drop in original_dataset.csv.
3. Replace HF auth token in Cell 2 with your own.
3. Ready to run!

Before we start, we need to make sure we have all the required dependencies installed and imported.

In [ ]:
# ============================================================
# CELL 1 — Install & Import All Dependencies
# ============================================================
!pip install -q datasets huggingface_hub
!pip install -q openai-whisper
!pip install -q sentence-transformers
!pip install -q pyarrow pandas tqdm soundfile librosa
!pip install -q ffmpeg-python
!pip install -q silero-vad
!apt-get install -qq ffmpeg

import os
import gc
import io
import time
import tempfile
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import soundfile as sf
import librosa
import torch
import whisper
from tqdm.notebook import tqdm
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from huggingface_hub import login
from sentence_transformers import SentenceTransformer
from google.colab import drive

print("✅ All packages installed and imported.")
print(f"🖥️  GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   Device: {torch.cuda.get_device_name(0)}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 11.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 46.3 MB/s eta 0:00:00
✅ All packages installed and imported.
🖥️  GPU available: False


All done! Next is setting up the Huggingface stream and mounting our Google Drive so that the outputs can be saved using our checkpoint system later on. We'll also define some variables for later use.

In [ ]:
# ============================================================
# CELL 2 — Configuration & Authentication
# ============================================================

# --- Hugging Face Auth ---
HF_TOKEN = "your_token_here"
login(token=HF_TOKEN, add_to_git_credential=False)
print("✅ Logged in to Hugging Face.")

METADATA_CSV_PATH = "/content/drive/MyDrive/vectors2vibes_output/original_dataset.csv"

# --- Google Drive ---
drive.mount("/content/drive")
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/vectors2vibes_output"
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
CHECKPOINT_PATH = os.path.join(DRIVE_OUTPUT_DIR, "processed_songs.parquet")
PROGRESS_LOG    = os.path.join(DRIVE_OUTPUT_DIR, "progress_log.txt")
print(f"✅ Google Drive mounted. Output → {DRIVE_OUTPUT_DIR}")

# --- Pipeline Settings ---
DATASET_NAME     = "vectors2vibes/vectors2vibes-discogs-audio"
CHECKPOINT_EVERY = 50          # Save every N songs
WHISPER_MODEL_BASE = "base"
WHISPER_MODEL_FALLBACK = "medium"
EMBED_MODEL      = "all-MiniLM-L6-v2"   # Fast & effective sentence embedding model
DEVICE           = "cuda" if torch.cuda.is_available() else "cpu"

print(f"\n⚙️  Config:")
print(f"   Whisper base model    : {WHISPER_MODEL_BASE}")
print(f"   Whisper fallback model: {WHISPER_MODEL_FALLBACK}")
print(f"   Embedding model : {EMBED_MODEL}")
print(f"   Device          : {DEVICE}")
print(f"   Checkpoint every: {CHECKPOINT_EVERY} songs")

✅ Logged in to Hugging Face.
Mounted at /content/drive
✅ Google Drive mounted. Output → /content/drive/MyDrive/vectors2vibes_output

⚙️  Config:
   Whisper base model    : base
   Whisper fallback model: medium
   Embedding model : all-MiniLM-L6-v2
   Device          : cpu
   Checkpoint every: 10 songs


Following this, we'll actually load the models we're going to be using. Whisper's base model will be used for lyric transcription, and all-MiniLM-L6-v2 as our sentence transformer, to generate the embeddings from the transcribed text.

The medium model will also be loaded as a fallback: when the base model produces low quality transcriptions (repetitive/too short/single sentence), it will get flagged and the transcription will be retried using the medium model. While it is more accurate, it is also much larger and thus takes more time to run. We will therefore only utilize it as a failsafe of sorts.


In [ ]:
# ============================================================
# CELL 3 — Load Models
# ============================================================

print("⏳ Loading Whisper base model...")
whisper_model = whisper.load_model(WHISPER_MODEL_BASE, device=DEVICE)
print(f"✅ Whisper '{WHISPER_MODEL_BASE}' loaded on {DEVICE}.")

print("\n⏳ Loading Whisper fallback model...")
whisper_model_fallback = whisper.load_model(WHISPER_MODEL_FALLBACK, device=DEVICE)
print(f"✅ Whisper '{WHISPER_MODEL_FALLBACK}' fallback loaded on {DEVICE}.")

print("\n⏳ Loading sentence-transformer model...")
embed_model = SentenceTransformer(EMBED_MODEL, device=DEVICE)
print(f"✅ SentenceTransformer '{EMBED_MODEL}' loaded.")

⏳ Loading Whisper base model...


100%|███████████████████████████████████████| 139M/139M [00:03<00:00, 39.5MiB/s]


✅ Whisper 'base' loaded on cpu.

⏳ Loading Whisper fallback model...


100%|█████████████████████████████████████| 1.42G/1.42G [01:40<00:00, 15.1MiB/s]


✅ Whisper 'medium' fallback loaded on cpu.

⏳ Loading sentence-transformer model...


...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

HTTP Error 504 thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/1_Pooling/config.json
Retrying in 1s [Retry 1/5].


config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ SentenceTransformer 'all-MiniLM-L6-v2' loaded.


We're almost done loading stuff! Lastly, we'll import the dataset CSV from Google Drive, which we mounted earlier in cell 2. We'll also do a quick check to ensure there are no duplicates, which was a problem in earlier iterations of the code.

In [ ]:
# ============================================================
# CELL 4 — Load Dataset & Determine Resume Point
# ============================================================

print("⏳ Loading metadata from Google Drive CSV...")
meta_df = pd.read_csv(METADATA_CSV_PATH)

# Preserve all original ID columns — do NOT overwrite them
ID_COLUMNS = ["id", "version_id", "release_id", "master_id"]
for col in ID_COLUMNS:
    if col in meta_df.columns:
        meta_df[col] = meta_df[col].astype(str)

# Validate — no NaN file_paths should ever exist in the source CSV
nan_count = meta_df['file_path'].isna().sum()
assert nan_count == 0, f"❌ {nan_count} NaN values found in 'file_path' column — fix the source CSV before continuing."

print(f"✅ Metadata loaded: {len(meta_df):,} tracks.")
print(f"   (includes {meta_df['file_path'].duplicated().sum():,} duplicate file_path rows)")

# --- Determine resume index from existing checkpoint ---
if os.path.exists(CHECKPOINT_PATH):
    existing_df = pd.read_parquet(CHECKPOINT_PATH, columns=["file_path"])
    already_processed_ids = set(existing_df["file_path"].tolist())
    print(f"🔁 Resuming — {len(already_processed_ids):,} songs already in checkpoint.")
else:
    already_processed_ids = set() # Initialize an empty set if no checkpoint
    print("🆕 No existing checkpoint found. Starting fresh.")

⏳ Loading metadata from Google Drive CSV...
✅ Metadata loaded: 24,695 tracks.
   (includes 0 duplicate file_path rows — kept intentionally)
🆕 No existing checkpoint found. Starting fresh.


While we're now done with loading, we're not completely finished setting up. We should first define some functions to ensure our main processing loop in cell 6 properly works.

These functions include the flagging for low quality, setting up Silero to detect the speech percentage of tracks, transcription (and fallback), embedding, and the save function of our checkpoint system.

In [ ]:
# ============================================================
# CELL 5 — Helper Functions
# ============================================================

# Quality thresholds — transcripts below either threshold trigger fallback
MIN_WORDS     = 10   # fewer than this = likely empty/instrumental/hallucination
MIN_SENTENCES = 2    # single sentence = likely a Whisper hallucination

def is_low_quality(text: str, duration_sec: float = None) -> bool:
    """Returns True if the transcript is empty, too short, or a single sentence.
    For short tracks (under 60s), the word threshold is scaled down proportionally."""
    if not text or not text.strip():
        return True
    words     = text.split()
    sentences = [s.strip() for s in text.replace("?", ".").replace("!", ".").split(".") if s.strip()]
    # Scale word threshold down for short tracks — a 20s track can't have 10 words of lyrics
    if duration_sec is not None and duration_sec < 60:
        word_threshold = max(3, int(MIN_WORDS * (duration_sec / 60)))
    else:
        word_threshold = MIN_WORDS
    return len(words) < word_threshold or len(sentences) < MIN_SENTENCES or has_repetition(text)

def has_repetition(text: str, threshold: float = 0.6) -> bool:
    """Returns True if most words are identical — likely a Whisper hallucination."""
    words = text.lower().split()
    if len(words) < 6:
        return False
    most_common_count = max(words.count(w) for w in set(words))
    return most_common_count / len(words) > threshold

# Minimum fraction of audio that must contain speech to attempt transcription
SPEECH_THRESHOLD = 0.005

# Load Silero VAD model once at definition time
_vad_model, _vad_utils = torch.hub.load(
    repo_or_dir="snakers4/silero-vad",
    model="silero_vad",
    force_reload=False,
    trust_repo=True,
)
_get_speech_timestamps = _vad_utils[0]

def _get_speech_fraction(audio_array: np.ndarray, sample_rate: int) -> float:
    """Returns the fraction of audio detected as speech. Returns 1.0 on VAD failure."""
    try:
        if audio_array.ndim > 1:
            audio_array = audio_array.mean(axis=-1)
        audio_array = audio_array.astype(np.float32)
        if sample_rate != 16000:
            audio_array = librosa.resample(audio_array, orig_sr=sample_rate, target_sr=16000)
        audio_tensor = torch.from_numpy(audio_array)
        timestamps   = _get_speech_timestamps(audio_tensor, _vad_model, sampling_rate=16000)
        if not timestamps:
            return 0.0
        speech_samples = sum(t["end"] - t["start"] for t in timestamps if t is not None)
        return speech_samples / len(audio_array)
    except Exception:
        return 1.0

def transcribe_audio(audio_array: np.ndarray, sample_rate: int, model=None) -> dict:
    """Resample to 16kHz (Whisper requirement) and transcribe with given model."""
    if model is None:
        model = whisper_model
    if audio_array.ndim > 1:
        audio_array = audio_array.mean(axis=-1)
    audio_array = audio_array.astype(np.float32)
    if sample_rate != 16000:
        audio_array = librosa.resample(audio_array, orig_sr=sample_rate, target_sr=16000)
    result = model.transcribe(audio_array, fp16=(DEVICE == "cuda"))
    if result is None:
        return {"text": "", "language": "unknown"}
    return {
        "text"    : result.get("text", "").strip(),
        "language": result.get("language", "unknown"),
    }

def transcribe_with_fallback(audio_array: np.ndarray, sample_rate: int) -> dict:
    """Transcribe with base model. If the result is low quality, retry with fallback.
    Returns the result dict with an added 'used_fallback' boolean."""
    duration_sec = len(audio_array) / sample_rate
    result = transcribe_audio(audio_array, sample_rate, model=whisper_model) or {"text": "", "language": "unknown"}
    used_fallback = False
    if is_low_quality(result["text"], duration_sec=duration_sec):
        fallback = transcribe_audio(audio_array, sample_rate, model=whisper_model_fallback) or {"text": "", "language": "unknown"}
        fallback_is_better = (
            not is_low_quality(fallback["text"], duration_sec=duration_sec)
            or len(fallback["text"]) > len(result["text"])
        )
        if fallback_is_better:
            result = fallback
            used_fallback = True
    result["used_fallback"] = used_fallback
    return result


def embed_text(text: str) -> list:
    """Generate a sentence embedding vector for the given text."""
    if not text:
        return [0.0] * 384
    vector = embed_model.encode(text, convert_to_numpy=True)
    return vector.tolist()


def save_checkpoint(records: list):
    """
    Append records to the master Parquet checkpoint on Drive.
    Deduplicates on write so a bad resume can never produce duplicates.
    """
    ID_COLS = ["id", "version_id", "release_id", "master_id"]

    new_df = pd.DataFrame(records)
    for col in ID_COLS:
        if col in new_df.columns:
            new_df[col] = new_df[col].astype(str)

    if os.path.exists(CHECKPOINT_PATH):
        existing_df = pd.read_parquet(CHECKPOINT_PATH)
        for col in ID_COLS:
            if col in existing_df.columns:
                existing_df[col] = existing_df[col].astype(str)
        combined_df = pd.concat([existing_df, new_df], ignore_index=True)
    else:
        combined_df = new_df

    combined_df = combined_df.drop_duplicates(subset="file_path", keep="last")
    combined_df.to_parquet(CHECKPOINT_PATH, index=False)
    with open(PROGRESS_LOG, "a") as f:
        f.write(f"{time.strftime('%Y-%m-%d %H:%M:%S')} — Saved {len(combined_df)} total songs\n")
    print(f"   💾 Checkpoint saved → {len(combined_df):,} total songs in Parquet.")


print("✅ Helper functions defined.")

Downloading: "https://github.com/snakers4/silero-vad/zipball/master" to /root/.cache/torch/hub/master.zip
✅ Helper functions defined.


Now that we're wrapped up with the setup, we can build our main processing loop! First, we'll ensure that it skips songs that have already been transcribed and embedded.

Then, we'll download each song from the Huggingface database, check whether they contain speech or are instrumentals, process them and save them to our checkpoint file every 50 songs. This ensures that any crashes, runtime disconnects and other interruptions will not result in significant setbacks every time.

In [ ]:
# ============================================================
# CELL 6 — Main Processing Loop (CSV-driven, no stream skipping)
# ============================================================
import soundfile as sf
from huggingface_hub import hf_hub_download

# --- Reload already-processed IDs fresh from disk ---
if os.path.exists(CHECKPOINT_PATH):
    _resume_df = pd.read_parquet(CHECKPOINT_PATH, columns=["file_path"])
    already_processed_ids = set(_resume_df["file_path"].tolist())
    print(f"🔁 Reloaded {len(already_processed_ids):,} already-processed IDs from checkpoint.")
else:
    already_processed_ids = set()
    print("🆕 No checkpoint found. Starting fresh.")

# --- Filter CSV to only unprocessed rows ---
remaining = meta_df[~meta_df["file_path"].isin(already_processed_ids)].reset_index(drop=True)
print(f"📋 {len(remaining):,} tracks remaining to process.\n")

# All CSV columns in order — record will mirror these exactly
CSV_COLUMNS = [
    "file_path", "id", "version_id", "release_id", "master_id",
    "track_title", "earliest_release", "discogs_released",
    "release_artist_names", "release_title", "release_genres",
    "release_styles", "country", "labels", "duration",
    "youtube_video", "webpage_url", "thumbnail", "thumbnail_path",
    "source_corpus", "ingest_batch",
]

batch_records   = []
total_processed = len(already_processed_ids)
errors          = 0

print(f"🚀 Starting pipeline. Already processed: {total_processed:,} songs.\n")

for idx, row in enumerate(tqdm(remaining.itertuples(index=False), total=len(remaining), desc="Processing songs")):
    file_path = row.file_path
    if not file_path or str(file_path).startswith("#"):
        errors += 1
        continue
    try:
        # --- Download audio file from HF ---
        local_path = hf_hub_download(
            repo_id=DATASET_NAME,
            filename=file_path,
            repo_type="dataset",
            token=HF_TOKEN,
        )

        # --- Load audio ---
        audio_array, sample_rate = sf.read(local_path, dtype="float32")
        if audio_array.ndim > 1:
            audio_array = audio_array.mean(axis=1)  # stereo → mono

        # --- Check for speech before transcribing ---
        speech_fraction = _get_speech_fraction(audio_array, sample_rate)
        transcription   = transcribe_with_fallback(audio_array, sample_rate)
        embedding       = embed_text(transcription["text"])

        # --- Build record: all CSV columns as-is, plus lyric_embeddings ---
        meta   = row._asdict()
        record = {col: meta.get(col, "") for col in CSV_COLUMNS}
        for col in ["id", "version_id", "release_id", "master_id"]:
            if col in record:
                record[col] = str(record[col])
        record["transcribed_lyrics"] = transcription["text"]
        record["lyric_language"]     = transcription["language"]
        record["whisper_fallback"]   = transcription["used_fallback"]
        record["vad_speech_fraction"] = round(float(speech_fraction), 4)
        record["lyric_embeddings"]   = embedding

        batch_records.append(record)
        already_processed_ids.add(file_path)
        total_processed += 1

        # --- Checkpoint every N songs ---
        if len(batch_records) >= CHECKPOINT_EVERY:
            print(f"\n📦 Checkpoint triggered at {total_processed:,} total songs processed...")
            save_checkpoint(batch_records)
            batch_records = []
            gc.collect()
            torch.cuda.empty_cache()



    except Exception as e:
        errors += 1
        print(f"\n⚠️  Error on '{file_path}': {e}")
    finally:
        # Always clean up — even if transcription or embedding failed
        if 'local_path' in locals() and os.path.exists(local_path):
            os.remove(local_path)

# --- Save any remaining records ---
if batch_records:
    print(f"\n📦 Saving final batch of {len(batch_records)} songs...")
    save_checkpoint(batch_records)

print(f"\n🎉 Pipeline complete!")
print(f"   ✅ Total processed : {total_processed:,}")
print(f"   ❌ Errors          : {errors:,}")
print(f"   💾 Output file     : {CHECKPOINT_PATH}")

🆕 No checkpoint found. Starting fresh.
📋 24,695 tracks remaining to process.

🚀 Starting pipeline. Already processed: 0 songs.



Processing songs:   0%|          | 0/24695 [00:00<?, ?it/s]

sZ/sZSpQwL2nks.ogg:   0%|          | 0.00/3.29M [00:00<?, ?B/s]

HTTP Error 504 thrown while requesting HEAD https://huggingface.co/datasets/vectors2vibes/vectors2vibes-discogs-audio/resolve/main/sZ/sZVXgac7Pcc.ogg
Retrying in 1s [Retry 1/5].
HTTP Error 504 thrown while requesting HEAD https://huggingface.co/datasets/vectors2vibes/vectors2vibes-discogs-audio/resolve/main/sZ/sZVXgac7Pcc.ogg
Retrying in 2s [Retry 2/5].
HTTP Error 504 thrown while requesting HEAD https://huggingface.co/datasets/vectors2vibes/vectors2vibes-discogs-audio/resolve/main/sZ/sZVXgac7Pcc.ogg
Retrying in 4s [Retry 3/5].
HTTP Error 504 thrown while requesting HEAD https://huggingface.co/datasets/vectors2vibes/vectors2vibes-discogs-audio/resolve/main/sZ/sZVXgac7Pcc.ogg
Retrying in 8s [Retry 4/5].
HTTP Error 504 thrown while requesting HEAD https://huggingface.co/datasets/vectors2vibes/vectors2vibes-discogs-audio/resolve/main/sZ/sZVXgac7Pcc.ogg
Retrying in 8s [Retry 5/5].
HTTP Error 504 thrown while requesting HEAD https://huggingface.co/datasets/vectors2vibes/vectors2vibes-discog


⚠️  Error on 'sZ/sZVXgac7Pcc.ogg': An error happened while trying to locate the file on the Hub and we cannot find the requested files in the local cache. Please check your connection and try again or make sure your Internet connection is on.


...

nJ/nJSZT6TXfik.ogg:   0%|          | 0.00/3.59M [00:00<?, ?B/s]

nJ/nJdzPrZjCCs.ogg:   0%|          | 0.00/4.04M [00:00<?, ?B/s]


📦 Checkpoint triggered at 10 total songs processed...
   💾 Checkpoint saved → 10 total songs in Parquet.


nJ/nJ7TDJfM_us.ogg:   0%|          | 0.00/10.2M [00:00<?, ?B/s]

KeyboardInterrupt: 

All done! As the very last step, we'll verify everything: columns, duplicates, embedding size (which should always be 384) and transcript quality, among others. We'll also print ten random rows so we can manually check whether all the columns contain the proper data.

In [ ]:
# ============================================================
# CELL 7 — Verification
# ============================================================

df_check = pd.read_parquet(CHECKPOINT_PATH)

# --- Size ---
print("📦 DATASET SIZE")
print(f"   Rows    : {len(df_check):,}")
print(f"   Columns : {len(df_check.columns)}")
print(f"   Column names: {list(df_check.columns)}\n")

# --- Duplicates ---
print("🔍 DUPLICATE CHECK")
n_dupes = df_check["file_path"].duplicated().sum()
n_unique = df_check["file_path"].nunique()
if n_dupes == 0:
    print(f"   ✅ No duplicates — {n_unique:,} unique file_paths")
else:
    print(f"   ❌ {n_dupes:,} duplicate rows found across {n_unique:,} unique file_paths")
    dupe_counts = df_check["file_path"].value_counts()
    dist = dupe_counts.value_counts().sort_index()
    for appearances, n_tracks in dist.items():
        flag = "✅" if appearances == 1 else "⚠️ "
        print(f"   {flag} appear {appearances}x → {n_tracks:,} track IDs")

# --- Embedding integrity ---
print("\n🔢 EMBEDDING CHECK")
embed_lengths = df_check["lyric_embeddings"].apply(lambda x: len(x) if hasattr(x, '__len__') else 0)
n_empty = (embed_lengths == 0).sum()
print(f"   Dimension : {embed_lengths.mode()[0]} (expected 384)")
print(f"   Min/Max   : {embed_lengths.min()} / {embed_lengths.max()}")
if n_empty > 0:
    print(f"   ⚠️  {n_empty:,} rows have empty embeddings")
else:
    print(f"   ✅ All embeddings present")

# --- Transcript quality ---
print("\n📝 TRANSCRIPT QUALITY")
empty    = df_check["transcribed_lyrics"].fillna("").str.strip().eq("").sum()
low_qual = df_check["transcribed_lyrics"].fillna("").apply(is_low_quality).sum()
fallback = df_check["whisper_fallback"].eq(True).sum() if "whisper_fallback" in df_check.columns else "N/A"
print(f"   Empty transcripts     : {empty:,} ({100*empty/len(df_check):.1f}%)")
print(f"   Low quality (<{MIN_WORDS} words or <{MIN_SENTENCES} sentences): {low_qual:,} ({100*low_qual/len(df_check):.1f}%)")
print(f"   Used fallback model   : {fallback}")

# --- Coverage ---
print("\n📊 COVERAGE")
for col in ["track_title", "release_artist_names", "release_genres"]:
    if col in df_check.columns:
        filled = df_check[col].replace("", pd.NA).notna().sum()
        print(f"   {col:30s}: {filled:,} / {len(df_check):,} filled ({100*filled/len(df_check):.1f}%)")

# --- 10 random rows (no embedding vector) ---
print("\n🎲 10 RANDOM ROWS")
preview_cols = [c for c in df_check.columns if c != "lyric_embeddings"]
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 30)
pd.set_option("display.width", 200)
print(df_check[preview_cols].sample(10).to_string(index=False))

📦 DATASET SIZE
   Rows    : 10
   Columns : 26
   Column names: ['file_path', 'id', 'version_id', 'release_id', 'master_id', 'track_title', 'earliest_release', 'discogs_released', 'release_artist_names', 'release_title', 'release_genres', 'release_styles', 'country', 'labels', 'duration', 'youtube_video', 'webpage_url', 'thumbnail', 'thumbnail_path', 'source_corpus', 'ingest_batch', 'transcribed_lyrics', 'lyric_language', 'whisper_fallback', 'vad_speech_fraction', 'lyric_embeddings']

🔍 DUPLICATE CHECK
   ✅ No duplicates — 10 unique file_paths

🔢 EMBEDDING CHECK
   Dimension : 384 (expected 384)
   Min/Max   : 384 / 384
   ✅ All embeddings present

📝 TRANSCRIPT QUALITY
   Empty transcripts     : 0 (0.0%)
   Low quality (<10 words or <2 sentences): 4 (40.0%)
   Used fallback model   : 2

📊 COVERAGE
   track_title                   : 10 / 10 filled (100.0%)
   release_artist_names          : 10 / 10 filled (100.0%)
   release_genres                : 10 / 10 filled (100.0%)

🎲 10 RANDOM R